In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import json

In [2]:
# Load Data
df = pd.read_csv('../Data/Cleaned/model_ready_data.csv')

In [3]:
# DEFINING FEATURES
features = [
    'rolling_3_minutes',
    'rolling_3_ict_index',
    'rolling_3_creativity',
    'rolling_3_influence',
    'rolling_3_threat',
    'rolling_3_total_points',
    'value'
]

In [4]:
# Rolling averages creating empty rows for GW 1-3
df = df.dropna(subset=features + ['target_next_gw_points'])

In [5]:
# SPLIT DATA (Historical vs Current Season)
train_seasons = ['2021-22', '2022-23', '2023-24', '2024-25']
test_season = '2025-26'

train_df = df[df['season'].isin(train_seasons)]
test_df = df[df['season'] == test_season]

X_train = train_df[features]
y_train = train_df['target_next_gw_points']

X_test = test_df[features]
y_test = test_df['target_next_gw_points']

In [6]:
# TRAIN MODEL
print("Training Model...")
model = LinearRegression()
model.fit(X_train, y_train)

Training Model...


LinearRegression()

In [7]:
# EVALUATE (RMSE & R2)
predictions = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f"✅ Model Training Complete.")
print(f"📉 Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"📊 R2 Score: {r2:.3f}")

✅ Model Training Complete.
📉 Root Mean Squared Error (RMSE): 2.16
📊 R2 Score: 0.196


In [8]:
# SHOW PREDICTIONS (Visual Check)
print("\n--- Example Predictions (2025-26 Season) ---")
test_df_viz = test_df.copy()
test_df_viz['predicted_xp'] = predictions
test_df_viz['error'] = test_df_viz['predicted_xp'] - test_df_viz['target_next_gw_points']
# Viewing top 5 predictions vs actuals
display_cols = ['name', 'GW', 'predicted_xp', 'target_next_gw_points', 'error']
print(test_df_viz[display_cols].head(10))


--- Example Predictions (2025-26 Season) ---
               name  GW  predicted_xp  target_next_gw_points     error
290    Aaron Hickey   1      0.086617                    1.0 -0.913383
293    Aaron Hickey   4      0.221945                    0.0  0.221945
294    Aaron Hickey   5      0.282519                    2.0 -1.717481
295    Aaron Hickey   6      0.160976                    0.0  0.160976
456  Aaron Ramsdale   1      2.501506                    0.0  2.501506
459  Aaron Ramsdale   4      0.381158                    0.0  0.381158
460  Aaron Ramsdale   5      0.381158                    0.0  0.381158
461  Aaron Ramsdale   6      0.381158                    0.0  0.381158
533    Aaron Ramsey   1      0.250251                    0.0  0.250251
536    Aaron Ramsey   4      0.217524                    0.0  0.217524


In [9]:
# SAVE MODEL
joblib.dump(model, '../Models/linear_reg_v1.pkl')
print("\n💾 Model saved successfully to ../Models/linear_reg_v1.pkl")


💾 Model saved successfully to ../Models/linear_reg_v1.pkl


In [10]:
# Defining metrics
metrics_data = {
    "rmse": round(rmse, 2),
    "r2": round(r2, 3),
    "notes": "Linear Regression model trained on historical FPL data up to 2024-25 season to predict next gameweek points."
}

In [11]:
# Saving metrics to JSON
with open('../metrics.json', 'w') as f:
    json.dump(metrics_data, f)

print("💾 Metrics saved successfully to ../metrics.json")


💾 Metrics saved successfully to ../metrics.json
